# Debye-Waller workflow with a machine-learned potential

This notebook computes thermal displacement matrices, and diffraction patterns
with Debye-Waller factors, with `DebyeWallerMaker`. The forces come from
MACE-OMAT-0-medium. For the polar materials, the Born charges and the
high-frequency dielectric tensor for the non-analytical correction (NAC) come
from MACE-Field.

| Part | Content | Needs | Time |
|---|---|---|---|
| A. Run the workflows | Si step by step, six materials, a q-point scan for Cu, a reference from the Materials Project, five potentials for 15 materials | MACE-Field, a pymatgen with anisotropic Debye-Waller factors, a Materials Project API key, matgl for the MatPES potentials | 3 to 4 min per small material on one A100 GPU |
| B. Our results | All figures and tables, drawn from the numbers of our runs | atomate2 | seconds |

To see the results, run only Part B. Select its first code cell and use "Run
Selected Cell and All Below". "Run All" also starts the workflow runs of Part A.

Contents:

- Background
- Installation
- Part A. Run the workflows
  - A1. Si step by step
  - A2. Six materials and a q-point scan for Cu
  - A3. Reference from the Materials Project
  - A4. Saving the numbers for Part B
  - A5. Comparing machine-learned potentials
  - A6. Saving the comparison for Part B
- Part B. Our results
  - B1. Thermal displacements against temperature
  - B2. Comparison with DFT and experiment at 300 K
  - B3. Anisotropic displacements of ZnO
  - B4. X-ray and neutron diffraction
  - B5. Electron diffraction
  - B6. q-point mesh
  - B7. Comparison of the potentials with DFT
  - B8. Imaginary modes
- Known limitations

## Background

The thermal displacement matrix $U$ of site $\kappa$ is a sum over the phonon
modes $q\nu$ of a q-point mesh with $N$ points:

$$U_{\alpha\beta}(\kappa) = \frac{\hbar}{2 N m_\kappa} \sum_{q\nu}
\frac{1 + 2 n_{q\nu}}{\omega_{q\nu}} e_\alpha(\kappa, q\nu) e^*_\beta(\kappa, q\nu)$$

where $m_\kappa$ is the mass of the site, $\omega_{q\nu}$ and $e(\kappa, q\nu)$ are
the frequency and the eigenvector of the mode, and $n_{q\nu}$ is its Bose-Einstein
occupation. The Debye-Waller factor of the site for the reciprocal lattice vector
$g$, with $|g| = 1/d_{hkl}$, is $\exp(-2\pi^2 g^T U g)$. For an isotropic
$U = U_\mathrm{iso} I$, this is $\exp(-B s^2)$ with $B = 8\pi^2 U_\mathrm{iso}$
and $s = 1/(2 d_{hkl})$.

**Step 1: Relaxation.** The structure is relaxed with the force field.

**Step 2: Phonons.** The phonon workflow computes the force constants from
displaced supercells, and for the polar materials the Born charges and the
dielectric tensor. It then computes $U$ of each site of the primitive cell from
0 to 1000 K in steps of 100 K. The q-point mesh has 30000 q-points per reciprocal
atom, rounded up to odd mesh numbers. The three acoustic modes at Gamma are left
out.

**Step 3: Diffraction patterns.** pymatgen computes the X-ray (XRD), neutron (ND)
and electron (TEM) diffraction patterns with the Debye-Waller factors at each
temperature, and once without them.

The same workflow runs with VASP as `atomate2.vasp.flows.debye_waller.DebyeWallerMaker`.

## Installation

```
pip install 'atomate2[phonons,ase]' mp-api
pip install git+https://github.com/mdi-group/mace-field.git@45d5c5fa7b40a155855b3d155df1760e36849e64
pip install git+https://github.com/hrushikesh-s/pymatgen.git@c6de1930577320a8ba12a8910893e2343d9e8465
```

MACE-Field is a fork of MACE that installs as `mace-torch` and replaces MACE. It
also runs MACE-OMAT-0-medium. The anisotropic Debye-Waller factors are not in a
pymatgen release yet, so pymatgen installs from git. Part B needs only atomate2.

## Part A. Run the workflows

Part A needs the installation above, the model file below and a Materials Project
API key.

### Potentials

`mace_mp(model="medium-omat-0")` downloads and caches MACE-OMAT-0-medium, which is
released under the Academic Software License. For the Born charges, download
`MACEField-MH-0-omat-dielectric.model` from the
[MACE-Field releases](https://github.com/mdi-group/mace-field/releases).

Set the path to the downloaded file in the next cell. Set `DEVICE` to `"cpu"`
without a GPU. We use `float64`, since the force constants come from small force
differences.

In [ ]:
import os
import warnings

# macOS: conda's llvm-openmp and torch's bundled libomp both load and the
# duplicate aborts the process. This must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
warnings.filterwarnings("ignore")

# Replace the path with the file you downloaded.
MACE_FIELD_MODEL = "/path/to/MACEField-MH-0-omat-dielectric.model"
DEVICE = "cuda"
CALC_KWARGS = {"model": "medium-omat-0", "device": DEVICE, "default_dtype": "float64"}

### A1. Si step by step

We run Si with MACE-OMAT-0-medium. Part B shows our results.

#### Structure

We take Si (mp-149) from the Materials Project. Set `MP_API_KEY` first.

In [ ]:
from mp_api.client import MPRester

if not os.environ.get("MP_API_KEY"):
    raise OSError("export MP_API_KEY before running this cell")

with MPRester() as mpr:
    structure = mpr.get_structure_by_material_id("mp-149")
structure

#### Workflow

`DebyeWallerMaker.from_force_field_name` sets one force field for the relaxation
and the phonons. The model name in `calculator_kwargs` selects the MACE weights.
`xrd_kwargs` and `nd_kwargs` go to pymatgen's `XRDCalculator` and `NDCalculator`.
We use Mo K$\alpha$ X-rays and 1 Å neutrons. These shorter wavelengths give more
peaks up to $2\theta = 90°$ than the defaults. All other settings are the
defaults.

In [ ]:
from atomate2.forcefields.flows.debye_waller import DebyeWallerMaker

maker = DebyeWallerMaker.from_force_field_name(
    "MACE-MP-0",
    calculator_kwargs=CALC_KWARGS,
    xrd_kwargs={"wavelength": "MoKa"},
    nd_kwargs={"wavelength": 1.0},
)
flow = maker.make(structure)
flow.draw_graph().show()

#### Run

The run took 4 min on one A100 GPU of Perlmutter.

In [ ]:
from jobflow import JobStore, run_locally
from maggma.stores import MemoryStore

job_store = JobStore(MemoryStore(), additional_stores={"data": MemoryStore()})
responses = run_locally(flow, store=job_store, create_folders=True, ensure_success=True)
doc = responses[flow.output.uuid][1].output

#### Output

The output is a `DebyeWallerDocument`. It holds $U$ of each site at each
temperature, the primitive cell, and the XRD, ND and TEM patterns. The next cell
prints $U_\mathrm{iso}$ at 300 K and the number of peaks and spots.

In [ ]:
import numpy as np

data = doc.thermal_displacement_data
temperatures = np.array(data.temperatures_thermal_displacements)
u = np.array(data.thermal_displacement_matrix)
i300 = int(np.argmin(np.abs(temperatures - 300)))
{
    "temperatures in K": temperatures.tolist(),
    "U_iso(300 K) in A^2": (np.trace(u[i300], axis1=1, axis2=2) / 3).round(6).tolist(),
    "XRD peaks": len(doc.xrd_pattern_static.x),
    "ND peaks": len(doc.nd_pattern_static.x),
    "TEM spots": len(doc.tem_pattern_static),
}

### A2. Six materials and a q-point scan for Cu

`run_dw` builds and runs one workflow in its own folder. The loop after it runs
Cu, W, Si, NaCl, MgO and ZnO. NaCl, MgO and ZnO get the Born charges and the
dielectric tensor from MACE-Field. The loop also runs Cu at five more q-point
densities for B6. We ran these 11 runs at once, one per A100 GPU, in 3 to 4 min
each. Set `RUN_ALL = True` to run them.

In [ ]:
from pymatgen.core import Structure

from atomate2.common.schemas.debye_waller import DebyeWallerDocument
from atomate2.forcefields.jobs import ForceFieldDielectricMaker

# name: Materials Project id, and whether the NAC is used
MATERIALS = {
    "Cu": ("mp-30", False),
    "W": ("mp-91", False),
    "Si": ("mp-149", False),
    "NaCl": ("mp-22862", True),
    "MgO": ("mp-1265", True),
    "ZnO": ("mp-2133", True),
}


def run_dw(
    structure: Structure,
    polar: bool,
    kppa: int,
    root_dir: str,
    force_field_name: str = "MACE-MP-0",
    calculator_kwargs: dict | None = None,
    **td_kwargs,
) -> DebyeWallerDocument:
    """Run the Debye-Waller workflow and return its output document.

    kppa is the q-point density of the thermal displacements. polar adds the
    MACE-Field Born charges and dielectric tensor for the NAC. td_kwargs are more
    settings of the thermal displacements, such as freq_min_thermal_displacements.
    """
    maker = DebyeWallerMaker.from_force_field_name(
        force_field_name,
        calculator_kwargs=CALC_KWARGS
        if calculator_kwargs is None
        else calculator_kwargs,
        xrd_kwargs={"wavelength": "MoKa"},
        nd_kwargs={"wavelength": 1.0},
    )
    if polar:
        maker.phonon_maker.born_maker = ForceFieldDielectricMaker(
            calculator_kwargs={"model": MACE_FIELD_MODEL, "device": DEVICE}
        )
    maker.phonon_maker.generate_frequencies_eigenvectors_kwargs = {
        "kpoint_density_thermal_displacements": kppa,
        **td_kwargs,
    }
    flow = maker.make(structure)
    os.makedirs(root_dir, exist_ok=True)
    responses = run_locally(
        flow,
        store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
        create_folders=True,
        root_dir=root_dir,
        ensure_success=True,
    )
    return responses[flow.output.uuid][1].output

In [ ]:
RUN_ALL = False
if RUN_ALL:
    with MPRester() as mpr:
        structures = {
            name: mpr.get_structure_by_material_id(mp_id)
            for name, (mp_id, _) in MATERIALS.items()
        }
    dw_docs = {
        name: run_dw(structures[name], polar, 30000, f"runs/{name}_30000")
        for name, (_, polar) in MATERIALS.items()
    }
    cu_scan = {
        kppa: run_dw(
            structures["Cu"], polar=False, kppa=kppa, root_dir=f"runs/Cu_{kppa}"
        )
        for kppa in (3000, 7000, 15000, 60000, 120000)
    }
    cu_scan[30000] = dw_docs["Cu"]

### A3. Reference from the Materials Project

The next cell computes $U$ from the force constants of the Materials Project's
Harmonic Phonon Database. These were computed with PBEsol and fitted with pheasy.
The cell uses the functions of the phonon workflow, so the NAC, the q-point mesh
and the temperatures are set as in A2.

In [ ]:
from phonopy import Phonopy
from pymatgen.io.phonopy import get_phonopy_structure, get_pmg_structure

from atomate2.common.schemas.phonons import (
    _get_thermal_displacement_data,
    _set_nac_params,
)


def mp_reference(
    mpr: MPRester, mp_id: str, nac: bool = True, **td_kwargs
) -> dict | None:
    """Return U from the pheasy force constants of the Materials Project.

    Returns None if the database has no pheasy force constants for mp_id.
    td_kwargs are passed on to _get_thermal_displacement_data.
    """
    summary = mpr.materials.summary.search(material_ids=[mp_id], fields=["phonon_IDs"])
    phonon_ids = (summary[0].phonon_IDs or {}).get("pheasy")
    if not phonon_ids:
        return None
    phonon_id = str(phonon_ids[0])
    mp_doc = mpr.materials.phonon.search(identifiers=[phonon_id], num_chunks=1)[0]
    phonon = Phonopy(
        get_phonopy_structure(mp_doc.structure),
        supercell_matrix=mp_doc.supercell_matrix,
        primitive_matrix=mp_doc.primitive_matrix,
    )
    phonon.force_constants = np.asarray(
        mpr.materials.phonon.get_forceconstants_from_phonon_id(phonon_id, "pheasy")
    )
    if nac:
        _set_nac_params(phonon, mp_doc.born, mp_doc.epsilon_static, 1e-4, "vasp")
    return {
        "phonon_id": phonon_id,
        # the cell of U
        "structure": get_pmg_structure(phonon.primitive),
        "thermal_displacement_data": _get_thermal_displacement_data(
            phonon, **td_kwargs
        ),
    }


if RUN_ALL:
    with MPRester() as mpr:
        mp_ref = {
            name: mp_reference(mpr, mp_id) for name, (mp_id, _) in MATERIALS.items()
        }

### A4. Saving the numbers for Part B

The next cell writes the numbers of A2 and A3 to
`tests/test_data/forcefields/debye_waller_tutorial/dw_results.json.gz`, which
Part B reads. It keeps the TEM spots of Si with at least 1% of the intensity of
the strongest spot. To plot your own runs, write them to another file and set
`RESULTS_FILE` in Part B.

In [ ]:
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path

from monty.serialization import dumpfn
from pymatgen.io.vasp import Kpoints

RESULTS_FILE = "../tests/test_data/forcefields/debye_waller_tutorial/dw_results.json.gz"


def sig(values: object, digits: int = 6) -> list[float]:
    """Round to a number of significant digits, to keep the file small."""
    return [float(f"{v:.{digits}g}") for v in np.ravel(values)]


def pattern_data(static: object, patterns: list) -> dict:
    """Return the peaks of a pattern without the factors and at each temperature.

    pymatgen leaves out the peaks below 1e-5 of the strongest one, so weak peaks
    at large angles are missing at high temperatures. Only the peaks found in every
    pattern are kept.
    """
    two_thetas = [list(pattern.x) for pattern in (static, *patterns)]
    common = set(two_thetas[0]).intersection(*two_thetas[1:])
    keep = [i for i, x in enumerate(two_thetas[0]) if x in common]
    intensity = []
    for x, pattern in zip(two_thetas[1:], patterns, strict=True):
        peaks = dict(zip(x, pattern.y, strict=True))
        intensity.append(sig([peaks[two_thetas[0][i]] for i in keep]))
    return {
        "two_theta": sig(np.asarray(static.x)[keep]),
        "d_hkl": sig(np.asarray(static.d_hkls)[keep]),
        "hkl": [list(static.hkls[i][0]["hkl"]) for i in keep],
        "intensity_static": sig(np.asarray(static.y)[keep]),
        "intensity": intensity,
    }


def u_data(data: dict) -> dict:
    """Return the Cartesian and CIF matrices U in A^2."""
    return {
        "U": np.round(data["thermal_displacement_matrix"], 6).tolist(),
        "Ucif": np.round(data["thermal_displacement_matrix_cif"], 6).tolist(),
    }


def export_results(dw_docs: dict, mp_ref: dict, cu_scan: dict, path: str) -> None:
    """Write the numbers that Part B plots."""
    materials = {
        name: {
            "material_id": MATERIALS[name][0],
            "structure": dw_doc.structure,
            "mace": u_data(dw_doc.thermal_displacement_data.model_dump()),
            "mp": u_data(mp_ref[name]["thermal_displacement_data"]),
            "xrd": pattern_data(dw_doc.xrd_pattern_static, dw_doc.xrd_patterns),
            "nd": pattern_data(dw_doc.nd_pattern_static, dw_doc.nd_patterns),
        }
        for name, dw_doc in dw_docs.items()
    }
    # TEM spots of Si with at least 1% of the intensity of the strongest spot
    si = dw_docs["Si"]
    spots = si.tem_pattern_static
    keep = [i for i, spot in enumerate(spots) if spot["intensity"] >= 0.01]
    tem = {
        "position": [spots[i]["position"] for i in keep],
        "hkl": [spots[i]["hkl"] for i in keep],
        "intensity_static": sig([spots[i]["intensity"] for i in keep]),
        "intensity": [sig([p[i]["intensity"] for i in keep]) for p in si.tem_patterns],
    }
    scan = {"kppa": [], "mesh": [], "U_iso": []}
    for kppa, dw_doc in sorted(cu_scan.items()):
        mesh = Kpoints.automatic_density(dw_doc.structure, kppa, force_gamma=True)
        u = np.array(dw_doc.thermal_displacement_data.thermal_displacement_matrix)
        scan["kppa"].append(kppa)
        # rounded up to odd numbers, as in the phonon flow
        scan["mesh"].append([int(n + 1 - n % 2) for n in mesh.kpts[0]])
        scan["U_iso"].append(sig(np.trace(u[:, 0], axis1=1, axis2=2) / 3))
    provenance = {
        "settings": "DebyeWallerMaker defaults with MACE-OMAT-0-medium in float64, "
        "xrd_kwargs {'wavelength': 'MoKa'} and nd_kwargs {'wavelength': 1.0}",
        "born_charges": f"{Path(MACE_FIELD_MODEL).name} for NaCl, MgO and ZnO",
        "reference": "pheasy force constants of the Materials Project's Harmonic "
        "Phonon Database",
        "versions": {
            package: version(package)
            for package in ("atomate2", "phonopy", "pymatgen", "mace-torch")
        },
        "date": datetime.now(UTC).date().isoformat(),
    }
    data = si.thermal_displacement_data
    dumpfn(
        {
            "provenance": provenance,
            "temperatures": data.temperatures_thermal_displacements,
            "materials": materials,
            "tem_si": tem,
            "cu_scan": scan,
        },
        path,
    )


if RUN_ALL:
    export_results(dw_docs, mp_ref, cu_scan, RESULTS_FILE)

### A5. Comparing machine-learned potentials

This section runs the same workflow with five potentials. Only the calculator
changes. We compare $U$ with the pheasy force constants of the Materials Project's
Harmonic Phonon Database, computed with PBEsol. The runs leave out the NAC, so the
potentials differ only in their force constants. The reference is computed with
and without the NAC.

We use the six materials of A2, the eight test materials of R. Keesey and AgBr
(mp-23231). AgBr has no pheasy force constants in the database, so only the
potentials give its $U$.

The pheasy force constants of all eight test materials have imaginary modes on
the q-point mesh. `imaginary_modes_thermal_displacements="exclude"` computes $U$
from the real modes only. We also compute the reference with "absolute", which
counts each imaginary mode as a real mode of the same absolute frequency, as in
the calculations of R. Keesey. Both need a positive
`freq_min_thermal_displacements`, because the term of a mode diverges as its
frequency goes to zero. We use 0.1 THz for all runs
and for the reference. For Cu, W and Si, $U$ with this cutoff agrees with A2 within
0.04%.

`write_u` writes $U$ of each site at each temperature to one file per material and
potential. To compare other potentials or materials, change `MLIPS` and
`MLIP_MATERIALS`. We ran the 75 runs in parallel on Perlmutter, one A100 GPU per
run.

In [ ]:
# name: force field name and calculator settings
MLIPS = {
    "MACE-OMAT-0": ("MACE-MP-0", CALC_KWARGS),
    "MACE-MATPES-PBE-0": ("MACE-MP-0", {**CALC_KWARGS, "model": "mace-matpes-pbe-0"}),
    "MACE-MATPES-r2SCAN-0": (
        "MACE-MP-0",
        {**CALC_KWARGS, "model": "mace-matpes-r2scan-0"},
    ),
    "MatPES-PBE": ("MatPES-PBE", {}),
    "MatPES-r2SCAN": ("MatPES-r2SCAN", {}),
}
MLIP_MATERIALS = {
    **{name: mp_id for name, (mp_id, _) in MATERIALS.items()},
    "UO3": "mp-375",
    "KTe": "mp-1554",
    "CaYAlO4": "mp-1227044",
    "Ba3In2O5F2": "mp-21445",
    "RbLaSiS4": "mp-18658",
    "Ba3Sr3(SbN)2": "mp-1214517",
    "TlCu3S2": "mp-13829",
    "CsHo2Cu3Se5": "mp-1192367",
    "AgBr": "mp-23231",
}
TD_KWARGS = {
    "imaginary_modes_thermal_displacements": "exclude",
    "freq_min_thermal_displacements": 0.1,
}


def write_u(doc: DebyeWallerDocument, path: str) -> None:
    """Write the Cartesian and CIF matrices U in A^2 of each site and temperature."""
    data = doc.thermal_displacement_data
    dumpfn(
        {
            "structure": doc.structure,
            "temperatures": data.temperatures_thermal_displacements,
            "U": data.thermal_displacement_matrix,
            "Ucif": data.thermal_displacement_matrix_cif,
            "imaginary_modes": data.imaginary_modes,
        },
        path,
    )


if RUN_ALL:
    os.makedirs("u_matrices", exist_ok=True)
    dw_mlip, pheasy_ref = {}, {}
    with MPRester() as mpr:
        for name, mp_id in MLIP_MATERIALS.items():
            structure = mpr.get_structure_by_material_id(mp_id)
            dw_mlip[name] = {}
            for mlip, (force_field_name, calculator_kwargs) in MLIPS.items():
                doc = run_dw(
                    structure,
                    polar=False,
                    kppa=30000,
                    root_dir=f"runs_mlip/{name}_{mlip}",
                    force_field_name=force_field_name,
                    calculator_kwargs=calculator_kwargs,
                    **TD_KWARGS,
                )
                write_u(doc, f"u_matrices/{name}_{mlip}.json.gz")
                dw_mlip[name][mlip] = doc
            pheasy_ref[name] = {
                "no_nac": mp_reference(mpr, mp_id, nac=False, **TD_KWARGS),
                "nac": mp_reference(mpr, mp_id, **TD_KWARGS),
                "absolute": mp_reference(
                    mpr,
                    mp_id,
                    nac=False,
                    **(
                        TD_KWARGS
                        | {"imaginary_modes_thermal_displacements": "absolute"}
                    ),
                ),
            }

### A6. Saving the comparison for Part B

The next cell writes the eigenvalues of $U$ of A5 to
`tests/test_data/forcefields/debye_waller_tutorial/dw_mlip_results.json.gz`. The
eigenvalues do not depend on the orientation of the cell, so they are compared
site by site. `StructureMatcher` puts the sites of each run in the order of the
sites of the reference. For KTe the cell also writes the XRD peaks of the pheasy
reference and of MACE-OMAT-0.

In [ ]:
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.phonon.thermal_displacements import ThermalDisplacementMatrices
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

MLIP_RESULTS_FILE = (
    "../tests/test_data/forcefields/debye_waller_tutorial/dw_mlip_results.json.gz"
)


def eigenvalues(u: object) -> list:
    """Eigenvalues of U in A^2, ascending, for each temperature and site."""
    return np.round(np.linalg.eigvalsh(np.asarray(u)), 6).tolist()


def site_order(reference: object, structure: object) -> list[int]:
    """Index of the site of structure that matches each site of reference."""
    matcher = StructureMatcher(
        primitive_cell=False, scale=True, attempt_supercell=False
    )
    mapping = matcher.get_mapping(structure, reference)
    if mapping is None:
        raise ValueError(f"{reference.formula}: the structures do not match")
    return [int(i) for i in mapping]


def xrd_data(structure: object, u: object) -> dict:
    """XRD peaks with Mo K-alpha without the factors and at each temperature."""
    xrd = XRDCalculator(wavelength="MoKa")
    patterns = []
    for u_t in u:
        tdm = ThermalDisplacementMatrices(
            ThermalDisplacementMatrices.get_reduced_matrix(np.array(u_t)),
            structure,
            temperature=None,
        )
        site_u = tdm.to_structure_with_site_properties_Ucif()
        patterns.append(xrd.get_pattern(site_u, scaled=False))
    return pattern_data(xrd.get_pattern(structure, scaled=False), patterns)


def export_mlip_results(dw_mlip: dict, pheasy_ref: dict, path: str) -> None:
    """Write the eigenvalues of U of each site, temperature, potential and reference.

    The eigenvalues do not depend on the orientation of the cell. The sites are in
    the order of the pheasy reference, or of MACE-OMAT-0 if there is none.
    """
    out = {}
    for name, docs in dw_mlip.items():
        refs = {
            label: pheasy_ref[name][key]
            for key, label in (
                ("no_nac", "pheasy"),
                ("nac", "pheasy, NAC"),
                ("absolute", "pheasy, absolute"),
            )
            if pheasy_ref[name][key] is not None
        }
        mace = docs["MACE-OMAT-0"]
        reference = refs["pheasy"]["structure"] if refs else mace.structure
        sga = SpacegroupAnalyzer(reference, symprec=1e-3)
        entry = {
            "material_id": MLIP_MATERIALS[name],
            "space_group": sga.get_space_group_symbol(),
            "species": [site.specie.symbol for site in reference],
            "wyckoff": list(sga.get_symmetry_dataset().wyckoffs),
            "eigenvalues": {},
            "imaginary_modes": {},
        }
        for label, ref in refs.items():
            data = ref["thermal_displacement_data"]
            u = data["thermal_displacement_matrix"]
            entry["eigenvalues"][label] = eigenvalues(u)
            entry["imaginary_modes"][label] = data["imaginary_modes"]
        for mlip, doc in docs.items():
            data = doc.thermal_displacement_data
            u = np.array(data.thermal_displacement_matrix)
            order = site_order(reference, doc.structure)
            entry["eigenvalues"][mlip] = eigenvalues(u[:, order])
            entry["imaginary_modes"][mlip] = data.imaginary_modes
        if name == "KTe":
            entry["xrd"] = {
                label: xrd_data(
                    reference,
                    refs[label]["thermal_displacement_data"][
                        "thermal_displacement_matrix"
                    ],
                )
                for label in ("pheasy", "pheasy, absolute")
            } | {
                "MACE-OMAT-0": xrd_data(
                    mace.structure,
                    mace.thermal_displacement_data.thermal_displacement_matrix,
                ),
            }
        out[name] = entry
    provenance = {
        "settings": "DebyeWallerMaker defaults with 30000 q-points per reciprocal "
        "atom, imaginary modes and modes below 0.1 THz left out, no NAC, "
        "xrd_kwargs {'wavelength': 'MoKa'}",
        "potentials": {
            mlip: f"{ff} {kwargs.get('model', '')}".strip()
            for mlip, (ff, kwargs) in MLIPS.items()
        },
        "reference": "pheasy force constants of the Materials Project's Harmonic "
        "Phonon Database, with and without the NAC, and with the imaginary modes "
        "as real modes of the same absolute frequency",
        "versions": {
            package: version(package)
            for package in ("atomate2", "phonopy", "pymatgen", "mace-torch", "matgl")
        },
        "date": datetime.now(UTC).date().isoformat(),
    }
    data = mace.thermal_displacement_data
    dumpfn(
        {
            "provenance": provenance,
            "temperatures": data.temperatures_thermal_displacements,
            "materials": out,
        },
        path,
    )


if RUN_ALL:
    export_mlip_results(dw_mlip, pheasy_ref, MLIP_RESULTS_FILE)

## Part B. Our results

Part B draws all figures and tables from the numbers of our runs. It needs only
atomate2 and runs in seconds. The next cell loads the numbers of A4 and prints
their settings and package versions. B7 loads the numbers of A6.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from monty.serialization import loadfn

RESULTS_FILE = "../tests/test_data/forcefields/debye_waller_tutorial/dw_results.json.gz"
results = loadfn(RESULTS_FILE)
temperatures = np.array(results["temperatures"])
materials = results["materials"]
i300 = int(np.argmin(np.abs(temperatures - 300)))
results["provenance"]

### B1. Thermal displacements against temperature

The next cell plots $U_\mathrm{iso}$, one third of the trace of $U$, of each
element against temperature. Solid lines are MACE-OMAT-0-medium, dashed lines the
reference from the Materials Project (MP DFT).

In [ ]:
def u_iso(entry: dict, source: str) -> dict[str, np.ndarray]:
    """Return U_iso in A^2 at each temperature for each element of a material.

    source is "mace" or "mp". The sites of one element are averaged.
    """
    species = [site.specie.symbol for site in entry["structure"]]
    iso = np.trace(np.array(entry[source]["U"]), axis1=2, axis2=3) / 3
    result = {}
    for element in dict.fromkeys(species):
        sites = [i for i, s in enumerate(species) if s == element]
        result[element] = iso[:, sites].mean(axis=1)
    return result


fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True)
for ax, (name, entry) in zip(axes.flat, materials.items(), strict=True):
    mace, mp = u_iso(entry, "mace"), u_iso(entry, "mp")
    for i, element in enumerate(mace):
        ax.plot(temperatures, mace[element], color=f"C{i}", label=f"{element}, MACE")
        ax.plot(
            temperatures,
            mp[element],
            color=f"C{i}",
            ls="--",
            label=f"{element}, MP DFT",
        )
    ax.set_title(name)
    ax.legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("Temperature (K)")
for ax in axes[:, 0]:
    ax.set_ylabel("$U_\\mathrm{iso}$ (Å$^2$)")
fig.tight_layout()
plt.show()

At 0 K, $U$ is the zero-point motion. At high temperature $U$ grows linearly with
temperature. From 300 to 1000 K, $U_\mathrm{iso}$ grows by a factor of 2.8 to 3.3,
close to the classical ratio of 10/3.

### B2. Comparison with DFT and experiment at 300 K

We compare $U_\mathrm{iso}$ at 300 K with these measurements:

- Si: neutron Pendellösung at 295.5 K, $B$ = 0.4761(17) Å² (Heacock et al.,
  [Science 373 (2021)](https://doi.org/10.1126/science.abc2794)).
- MgO: X-ray, $B$ = 0.30(1) Å² for Mg and 0.34(2) Å² for O (Lawrence,
  [Acta Cryst. A29, 94 (1973)](https://doi.org/10.1107/S0567739473000240)).
- ZnO: single-crystal X-ray at 293 K (Sowa and Ahsbahs,
  [J. Appl. Cryst. 39, 169 (2006)](https://doi.org/10.1107/S0021889805042457)).
  We use $U_\mathrm{iso} = (2 U_{11} + U_{33})/3$.
- NaCl: single-crystal X-ray at 295 K, Crystallography Open Database entry
  7132177 (Mettler et al., Chem. Commun. (2023)).
- Cu and W: $B$ at 293 K computed from measured phonon densities of states (Peng
  et al., Acta Cryst. A52, 456 (1996)). These are not direct measurements of $B$.

In [ ]:
# U_iso in A^2 at 293 to 295.5 K, from the references above
EXPERIMENT = {
    ("Cu", "Cu"): 0.5505 / (8 * np.pi**2),
    ("W", "W"): 0.1596 / (8 * np.pi**2),
    ("Si", "Si"): 0.4761 / (8 * np.pi**2),
    ("NaCl", "Na"): 0.0231,
    ("NaCl", "Cl"): 0.0204,
    ("MgO", "Mg"): 0.30 / (8 * np.pi**2),
    ("MgO", "O"): 0.34 / (8 * np.pi**2),
    ("ZnO", "Zn"): (2 * 0.0068 + 0.0061) / 3,
    ("ZnO", "O"): (2 * 0.007 + 0.008) / 3,
}

rows = {}
for (name, element), u_exp in EXPERIMENT.items():
    u_mace = u_iso(materials[name], "mace")[element][i300]
    u_mp = u_iso(materials[name], "mp")[element][i300]
    rows[(name, element)] = {
        "MACE": round(u_mace, 5),
        "MP DFT": round(u_mp, 5),
        "experiment": round(u_exp, 5),
        "MACE vs exp. (%)": round(100 * (u_mace / u_exp - 1), 1),
        "MP DFT vs exp. (%)": round(100 * (u_mp / u_exp - 1), 1),
    }
pd.DataFrame(rows).T

| Material | Element | MACE | MP DFT | Experiment | MACE vs exp. | MP DFT vs exp. |
|---|---|---|---|---|---|---|
| Cu | Cu | 0.00721 | 0.00684 | 0.00697 | +3.4% | −1.8% |
| W | W | 0.00219 | 0.00211 | 0.00202 | +8.5% | +4.2% |
| Si | Si | 0.00663 | 0.00685 | 0.00603 | +9.9% | +13.6% |
| NaCl | Na | 0.02312 | 0.02044 | 0.02310 | +0.1% | −11.5% |
| NaCl | Cl | 0.01891 | 0.01768 | 0.02040 | −7.3% | −13.3% |
| MgO | Mg | 0.00419 | 0.00393 | 0.00380 | +10.3% | +3.3% |
| MgO | O | 0.00420 | 0.00395 | 0.00431 | −2.4% | −8.2% |
| ZnO | Zn | 0.00759 | 0.00712 | 0.00657 | +15.5% | +8.4% |
| ZnO | O | 0.00707 | 0.00666 | 0.00733 | −3.6% | −9.2% |

$U_\mathrm{iso}$ at 300 K in Å².

MACE-OMAT-0-medium gives a 4% to 13% larger $U_\mathrm{iso}$ than the PBEsol
reference, except for Si, where it is 3% smaller. Both are within 16% of
experiment. The mean absolute difference from experiment is 6.8% for MACE and
8.2% for MP DFT. At the default q-point density, $U_\mathrm{iso}$ of Cu is 2.8%
below its converged value (B6).

### B3. Anisotropic displacements of ZnO

In wurtzite ZnO, $U$ has two independent components, $U_{11}$ along $a$ and
$U_{33}$ along $c$. The next cell compares them with the measurement of Sowa and
Ahsbahs at 293 K.

In [ ]:
# U11 and U33 in A^2 at 293 K (Sowa and Ahsbahs, 2006)
ZNO_EXPERIMENT = {"Zn": (0.0068, 0.0061), "O": (0.007, 0.008)}

species = [site.specie.symbol for site in materials["ZnO"]["structure"]]
rows = {}
for element, (u11_exp, u33_exp) in ZNO_EXPERIMENT.items():
    site = species.index(element)
    for source, label in (("mace", "MACE"), ("mp", "MP DFT")):
        u_cif = np.array(materials["ZnO"][source]["Ucif"])[i300, site]
        rows[(element, label)] = {"U11": u_cif[0, 0], "U33": u_cif[2, 2]}
    rows[(element, "experiment")] = {"U11": u11_exp, "U33": u33_exp}
anisotropy = pd.DataFrame(rows).T
anisotropy["U33/U11"] = anisotropy["U33"] / anisotropy["U11"]
anisotropy.round(4)

| Element | Source | $U_{11}$ | $U_{33}$ | $U_{33}/U_{11}$ |
|---|---|---|---|---|
| Zn | MACE | 0.0074 | 0.0079 | 1.06 |
| Zn | MP DFT | 0.0070 | 0.0074 | 1.06 |
| Zn | experiment | 0.0068(4) | 0.0061(4) | 0.90 |
| O | MACE | 0.0070 | 0.0073 | 1.04 |
| O | MP DFT | 0.0065 | 0.0069 | 1.05 |
| O | experiment | 0.007(2) | 0.008(2) | 1.14 |

$U$ at 300 K in Å².

Both calculations give $U_{33}$ larger than $U_{11}$ by 4% to 6% for both
elements. The measurement gives $U_{33}$ 10% smaller than $U_{11}$ for Zn. For O,
its uncertainty is larger than the anisotropy.

The eigenvalues of $U$ give the mean squared displacement along the three principal
axes of the ellipsoid of a site. The next cell plots them against temperature for
ZnO. For both sites the two smaller eigenvalues are equal and lie in the
hexagonal plane. The largest is along $c$.

In [ ]:
_, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, element in zip(axes, ("Zn", "O"), strict=True):
    site = species.index(element)
    for source, label, ls in (("mace", "MACE", "-"), ("mp", "MP DFT", "--")):
        values = np.linalg.eigvalsh(np.array(materials["ZnO"][source]["U"])[:, site])
        for k, axis in enumerate(("min", "mid", "max")):
            ax.plot(
                temperatures,
                values[:, k],
                color=f"C{k}",
                ls=ls,
                label=f"$U_\\mathrm{{{axis}}}$, {label}",
            )
    ax.set_title(f"{element} in ZnO")
    ax.set_xlabel("Temperature (K)")
axes[0].set_ylabel("Eigenvalue of $U$ (Å$^2$)")
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

MACE gives eigenvalues 5% to 7% larger than MP DFT for both sites at 300 K. The
ratio of the eigenvalue along $c$ to the in-plane one is 1.06 for Zn with both
sources. For O it is 1.04 with MACE and 1.05 with MP DFT.

### B4. X-ray and neutron diffraction

The next cell plots the XRD pattern of Si without the Debye-Waller factors and at
300 and 1000 K. The three sets of peaks are shifted by 0.3° for clarity. The
intensities are not scaled.

In [ ]:
def plot_peaks(pattern: dict, title: str) -> None:
    """Plot the peaks without the factors and at 300 and 1000 K."""
    two_theta = np.array(pattern["two_theta"])
    _, ax = plt.subplots(figsize=(10, 3.5))
    for i, (intensity, label) in enumerate(
        (
            (pattern["intensity_static"], "without factors"),
            (pattern["intensity"][i300], "300 K"),
            (pattern["intensity"][-1], "1000 K"),
        )
    ):
        shift = 0.3 * (i - 1)
        ax.vlines(two_theta + shift, 0, intensity, label=label, color=f"C{i}")
    ax.set_xlabel("2$\\theta$ (°)")
    ax.set_ylabel("Intensity (not scaled)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_peaks(materials["Si"]["xrd"], "Si, XRD with Mo K$\\alpha$")

For a material with one element, every peak gets the same factor $\exp(-2Bs^2)$
in its intensity. So $\ln(I_T/I_0) = -B/(2 d_{hkl}^2)$ is a straight line in
$1/d_{hkl}^2$, where $I_T$ and $I_0$ are the intensities at temperature $T$ and
without the factors. The next cell fits this line at 300 K for Cu, W and Si and
compares $B$ with $8\pi^2 U_\mathrm{iso}$.

In [ ]:
_, ax = plt.subplots(figsize=(6, 4))
rows = {}
for i, name in enumerate(("Cu", "W", "Si")):
    for kind, marker in (("xrd", "o"), ("nd", "x")):
        pattern = materials[name][kind]
        inv_d2 = 1 / np.array(pattern["d_hkl"]) ** 2
        intensity = np.array(pattern["intensity"][i300])
        log_ratio = np.log(intensity / np.array(pattern["intensity_static"]))
        slope = np.polyfit(inv_d2, log_ratio, 1)[0]
        rows[(name, kind.upper())] = {
            "B from the fit (A^2)": round(-2 * slope, 4),
            "8 pi^2 U_iso (A^2)": round(
                8 * np.pi**2 * u_iso(materials[name], "mace")[name][i300], 4
            ),
        }
        label = f"{name}, {kind.upper()}"
        ax.plot(inv_d2, log_ratio, marker, color=f"C{i}", label=label)
ax.set_xlabel("$1/d_{hkl}^2$ (Å$^{-2}$)")
ax.set_ylabel("$\\ln(I_{300\\,K}/I_0)$")
ax.legend()
plt.tight_layout()
plt.show()
pd.DataFrame(rows).T

| Material | XRD | ND | $8\pi^2 U_\mathrm{iso}$ |
|---|---|---|---|
| Cu | 0.5690 | 0.5690 | 0.5690 |
| W | 0.1731 | 0.1731 | 0.1731 |
| Si | 0.5232 | 0.5232 | 0.5232 |

$B$ at 300 K in Å², from the fit to the XRD and ND peaks.

The fits give back $8\pi^2 U_\mathrm{iso}$ to four digits for both XRD and ND.
In NaCl, MgO and ZnO, each element has its own factor. Their $\ln(I_T/I_0)$ is
then not a straight line in $1/d_{hkl}^2$.

The next cell plots $\log_{10}(I_T/I_0)$ of each XRD peak of ZnO against
$2\theta$, at each temperature. The factors take away more intensity at larger
angles and at higher temperatures.

In [ ]:
pattern = materials["ZnO"]["xrd"]
two_theta = np.array(pattern["two_theta"])
static = np.array(pattern["intensity_static"])
keep = static > 1e-3 * static.max()
# temperature from light to dark blue
blues = plt.get_cmap("Blues")
t_cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "temperature", [blues(0.3), blues(1.0)]
)
_, ax = plt.subplots(figsize=(8, 3.5))
for t, intensity in zip(temperatures, pattern["intensity"], strict=True):
    ratio = np.log10(np.array(intensity)[keep] / static[keep])
    ax.plot(two_theta[keep], ratio, "o", ms=2.5, color=t_cmap(t / 1000))
sm = plt.cm.ScalarMappable(cmap=t_cmap, norm=plt.Normalize(0, 1000))
plt.colorbar(sm, ax=ax, label="Temperature (K)")
ax.set_xlabel("2$\\theta$ (°)")
ax.set_ylabel("$\\log_{10}(I_T/I_0)$")
ax.set_title("ZnO, XRD with Mo K$\\alpha$, MACE")
plt.tight_layout()
plt.show()

At 300 K the factors take away up to a factor of 3.3 of the intensity of a peak.
At 1000 K they take away up to 1.7 orders of magnitude, at 89°.

### B5. Electron diffraction

`TEMCalculator` puts the beam along the third lattice vector of the primitive cell
by default. For Si this lattice vector has the length $a/\sqrt{2}$, where $a$ is
the cubic lattice parameter. It is therefore a ⟨110⟩ direction of the diamond
lattice. The next cell checks this length.

In [ ]:
si = materials["Si"]["structure"]
{
    "cubic a from the volume (A)": round((4 * si.volume) ** (1 / 3), 4),
    "sqrt(2) |a3| (A)": round(float(np.sqrt(2) * si.lattice.c), 4),
}

The next cell plots the spots without the Debye-Waller factors and at 1000 K, with
an area proportional to the intensity. Each pattern is normalized to its
strongest spot, the {111} spots.

In [ ]:
tem = results["tem_si"]
positions = np.array(tem["position"])
_, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, intensity, title in (
    (axes[0], tem["intensity_static"], "Si, without factors"),
    (axes[1], tem["intensity"][-1], "Si, 1000 K"),
):
    ax.scatter(positions[:, 0], positions[:, 1], s=300 * np.array(intensity))
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlabel("x (cm)")
axes[0].set_ylabel("y (cm)")
plt.show()

The structure factor of each spot gets the factor $\exp(-2\pi^2 U_\mathrm{iso}/d^2)$,
and the intensity its square. After the normalization, the intensity of a spot at
temperature $T$ is its intensity without the factors times
$\exp(-4\pi^2 U_\mathrm{iso} (1/d^2 - 1/d_{111}^2))$. The next cell lists the
strongest spot of each family, with $h^2 + k^2 + l^2 = (a/d)^2$ in cubic indices,
and checks this relation at 1000 K.

In [ ]:
d_hkl = np.array([si.lattice.d_hkl(hkl) for hkl in tem["hkl"]])
static = np.array(tem["intensity_static"])
hot = np.array(tem["intensity"][-1])
u_1000 = u_iso(materials["Si"], "mace")["Si"][-1]
d_111 = d_hkl[np.argmax(static)]
expected = static * np.exp(-4 * np.pi**2 * u_1000 * (1 / d_hkl**2 - 1 / d_111**2))
a_cubic = (4 * si.volume) ** (1 / 3)

rows = {}
for i in np.argsort(-static):
    family = round((a_cubic / d_hkl[i]) ** 2)
    if family not in rows and len(rows) < 6:
        rows[family] = {
            "d (A)": round(d_hkl[i], 4),
            "without factors": round(static[i], 4),
            "1000 K": round(hot[i], 4),
            "expected at 1000 K": round(expected[i], 4),
        }
pd.DataFrame(rows).T.rename_axis("h^2+k^2+l^2")

| $h^2+k^2+l^2$ | $d$ (Å) | Without factors | 1000 K |
|---|---|---|---|
| 3 | 3.1314 | 1.0 | 1.0 |
| 8 | 1.9176 | 0.6622 | 0.5763 |
| 16 | 1.3559 | 0.2467 | 0.1719 |
| 11 | 1.6353 | 0.2137 | 0.1711 |
| 24 | 1.1071 | 0.1369 | 0.0764 |
| 19 | 1.2443 | 0.0957 | 0.0613 |

Normalized TEM intensities of Si.

The stored intensities agree with this relation within 0.02%. At 1000 K the {422}
spots lose 44% of their intensity relative to the {111} spots.

### B6. q-point mesh

The sum over the mesh leaves out the three acoustic modes at Gamma, so the cell
around Gamma is missing from the sum. $U$ is therefore too small, by an error that
falls as $1/N$ for an $N \times N \times N$ mesh. The next cell plots
$U_\mathrm{iso}$ of Cu at 300 K against $1/N$ for the six densities of A2. A
straight line through the two densest meshes gives the converged value $U_\infty$.

In [ ]:
scan = results["cu_scan"]
mesh = np.array([m[0] for m in scan["mesh"]])
u_300 = np.array([u[i300] for u in scan["U_iso"]])
slope, u_inf = np.polyfit(1 / mesh[-2:], u_300[-2:], 1)

_, ax = plt.subplots(figsize=(6, 4))
ax.plot(1 / mesh, u_300, "o", label="Cu, MACE")
x = np.linspace(0, 1 / mesh[0], 50)
ax.plot(x, u_inf + slope * x, "--", label="line through the two densest meshes")
ax.set_xlabel("$1/N$")
ax.set_ylabel("$U_\\mathrm{iso}$(300 K) (Å$^2$)")
ax.legend()
plt.show()

pd.DataFrame(
    {
        "q-points per reciprocal atom": scan["kppa"],
        "mesh": [f"{n}x{n}x{n}" for n in mesh],
        "U_iso (A^2)": u_300.round(6),
        "error (%)": (100 * (u_300 / u_inf - 1)).round(1),
    }
)

| q-points per reciprocal atom | Mesh | $U_\mathrm{iso}$ (Å²) | Error |
|---|---|---|---|
| 3000 | 15x15x15 | 0.006987 | −5.8% |
| 7000 | 19x19x19 | 0.007076 | −4.6% |
| 15000 | 25x25x25 | 0.007157 | −3.5% |
| 30000 | 31x31x31 | 0.007207 | −2.8% |
| 60000 | 39x39x39 | 0.007249 | −2.2% |
| 120000 | 49x49x49 | 0.007283 | −1.8% |

$U_\mathrm{iso}$ of Cu at 300 K, and its error against $U_\infty$ = 0.00741 Å².

The line through the two densest meshes predicts the four coarser meshes within
0.03%. The default density of 30000 gives $U_\mathrm{iso}$ 2.8% below $U_\infty$.
For a converged value, run two dense meshes and extrapolate in the same way.

### B7. Comparison of the potentials with DFT

The next cell loads the numbers of A5 and A6. For each site it holds the three
eigenvalues of $U$ at each temperature, for the five potentials and for the pheasy
reference with and without the NAC.

In [ ]:
MLIP_RESULTS_FILE = (
    "../tests/test_data/forcefields/debye_waller_tutorial/dw_mlip_results.json.gz"
)
mlip_results = loadfn(MLIP_RESULTS_FILE)
mlip_materials = mlip_results["materials"]
POTENTIALS = list(mlip_results["provenance"]["potentials"])
mlip_results["provenance"]

The next cell lists the materials whose q-point mesh has imaginary modes, for each
source. $U$ of these is computed from the real modes above 0.1 THz only, and is an
estimate.

In [ ]:
pd.DataFrame(
    {
        name: {source: flag or "" for source, flag in entry["imaginary_modes"].items()}
        for name, entry in mlip_materials.items()
    }
).T.fillna("no reference")

With the pheasy force constants, all eight test materials have imaginary modes on
the mesh. KTe has none with any of the five potentials, and Ba3Sr3(SbN)2 has none
with four of them. UO3, CaYAlO4, Ba3In2O5F2 and CsHo2Cu3Se5 have imaginary modes
with all five. MatPES-PBE has imaginary modes for AgBr.

The next cell plots the eigenvalues of each potential against those of the pheasy
reference without the NAC, for all sites at all temperatures. Only materials
without imaginary modes in either source are plotted. The numbers in each panel
are the mean absolute error (MAE) and the mean signed error (MSE) over these
points.

In [ ]:
def paired(potential: str, stable_only: bool) -> tuple[np.ndarray, ...]:
    """Eigenvalues of the pheasy reference and of one potential, and temperatures."""
    ref, calc, temp = [], [], []
    for entry in mlip_materials.values():
        values = entry["eigenvalues"]
        if "pheasy" not in values or potential not in values:
            continue
        flags = entry["imaginary_modes"]
        if stable_only and (flags["pheasy"] or flags[potential]):
            continue
        ref_t = np.array(values["pheasy"])
        ref.append(ref_t.ravel())
        calc.append(np.array(values[potential]).ravel())
        temp.append(np.repeat(temperatures, ref_t[0].size))
    return np.concatenate(ref), np.concatenate(calc), np.concatenate(temp)


pairs = {potential: paired(potential, stable_only=True) for potential in POTENTIALS}
top = 1.05 * max(max(ref.max(), calc.max()) for ref, calc, _ in pairs.values())
fig, axes = plt.subplots(
    1,
    len(POTENTIALS),
    figsize=(16, 3.8),
    sharex=True,
    sharey=True,
    layout="constrained",
)
for ax, (potential, (ref, calc, temp)) in zip(axes, pairs.items(), strict=True):
    points = ax.scatter(ref, calc, c=temp, cmap=t_cmap, vmin=0, vmax=1000, s=10)
    ax.plot([0, top], [0, top], color="0.5", lw=1)
    error = calc - ref
    ax.text(
        0.05,
        0.95,
        f"MAE {np.abs(error).mean():.1e} Å$^2$\nMSE {error.mean():+.1e} Å$^2$",
        transform=ax.transAxes,
        va="top",
        fontsize=8,
    )
    ax.set_title(potential, fontsize=10)
    ax.set_xlabel("pheasy (Å$^2$)")
    ax.set_aspect("equal")
axes[0].set_xlim(0, top)
axes[0].set_ylim(0, top)
axes[0].set_ylabel("Potential (Å$^2$)")
fig.colorbar(points, ax=axes, label="Temperature (K)", shrink=0.8)
plt.show()

The next cell gives the same errors at 300 K, as a share of the reference, for the
stable materials and for all materials. The relative error is the median of
$(U_\mathrm{potential} - U_\mathrm{pheasy})/U_\mathrm{pheasy}$ over the
eigenvalues.

In [ ]:
rows = {}
for potential in POTENTIALS:
    for stable_only, label in ((True, "stable"), (False, "all")):
        ref, calc, temp = paired(potential, stable_only)
        at_300 = temp == temperatures[i300]
        relative = (calc[at_300] - ref[at_300]) / ref[at_300]
        rows[(potential, label)] = {
            "MAE (A^2)": np.abs(calc[at_300] - ref[at_300]).mean(),
            "MSE (A^2)": (calc[at_300] - ref[at_300]).mean(),
            "median relative error (%)": 100 * np.median(relative),
            "eigenvalues": at_300.sum(),
        }
pd.DataFrame(rows).T.round(
    {"MAE (A^2)": 5, "MSE (A^2)": 5, "median relative error (%)": 1}
)

| Potential | MAE, stable (Å²) | MSE, stable (Å²) | Median error, stable | MAE, all (Å²) | Median error, all |
|---|---|---|---|---|---|
| MACE-OMAT-0 | 0.00063 | +0.00055 | +6.6% | 0.00380 | +10.3% |
| MACE-MATPES-PBE-0 | 0.00192 | +0.00153 | +19.8% | 0.00654 | +23.9% |
| MACE-MATPES-r2SCAN-0 | 0.00091 | −0.00039 | −2.4% | 0.00358 | +1.9% |
| MatPES-PBE | 0.00229 | +0.00229 | +26.9% | 0.00585 | +22.8% |
| MatPES-r2SCAN | 0.00080 | +0.00080 | +13.4% | 0.00583 | +18.8% |

Errors of the eigenvalues at 300 K against pheasy without the NAC. The stable set
has 36 eigenvalues (the six materials of A2), the full set 387.

For the stable materials, MACE-OMAT-0 has the smallest MAE and MACE-MATPES-r2SCAN-0
the smallest median error. The two PBE potentials give $U$ 20% to 27% larger than
the PBEsol reference. PBE gives longer bonds and softer phonons than PBEsol. The errors for all materials are several times
larger than for the stable ones, because $U$ of the unstable materials depends on
the modes next to the imaginary ones (see below).

The next cell plots the median and the interquartile range (IQR) of each of the
three eigenvalues over all sites of the 14 materials with a pheasy reference,
against temperature, for the reference and each potential.

In [ ]:
with_reference = [
    name for name, entry in mlip_materials.items() if "pheasy" in entry["eigenvalues"]
]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True, layout="constrained")
for k, (ax, axis) in enumerate(zip(axes, ("min", "mid", "max"), strict=True)):
    for i, source in enumerate(["pheasy", *POTENTIALS]):
        # eigenvalue k of every site, shape (temperature, site)
        values = np.concatenate(
            [
                np.array(mlip_materials[name]["eigenvalues"][source])[:, :, k]
                for name in with_reference
            ],
            axis=1,
        )
        q1, median, q3 = np.percentile(values, [25, 50, 75], axis=1)
        color = "black" if source == "pheasy" else f"C{i - 1}"
        ax.plot(temperatures, median, "o-", ms=3, color=color, label=source)
        ax.fill_between(temperatures, q1, q3, color=color, alpha=0.12)
    ax.set_title(f"$U_\\mathrm{{{axis}}}$")
    ax.set_xlabel("Temperature (K)")
axes[0].set_ylabel("Eigenvalue of $U$ (Å$^2$), median and IQR over sites")
axes[2].legend(fontsize=8)
plt.show()

At 300 K, $U_\mathrm{min}$ of every potential is 11% to 36% larger than that of
pheasy. The median $U_\mathrm{max}$ is 0.022 Å² for pheasy and MACE-OMAT-0, 0.018 Å²
for MACE-MATPES-r2SCAN-0 and 0.029 Å² for MACE-MATPES-PBE-0. The IQR is wide,
because the 129 sites range from W to KTe.

The next cell compares $U_\mathrm{iso}$ of each potential at 300 K with the
measurements of B2, the mean of the three eigenvalues over the sites of each
element. The potentials run without the NAC, unlike MACE in B2.

In [ ]:
rows = {}
for (name, element), u_exp in EXPERIMENT.items():
    entry = mlip_materials[name]
    sites = np.array(entry["species"]) == element
    rows[(name, element)] = {
        source: 100
        * (np.array(entry["eigenvalues"][source])[i300][sites].mean() / u_exp - 1)
        for source in ["pheasy", *POTENTIALS]
    }
vs_experiment = pd.DataFrame(rows).T
vs_experiment.loc[("mean absolute", ""), :] = vs_experiment.abs().mean()
vs_experiment.round(1)

| Material | Element | pheasy | MACE-OMAT-0 | MACE-MATPES-PBE-0 | MACE-MATPES-r2SCAN-0 | MatPES-PBE | MatPES-r2SCAN |
|---|---|---|---|---|---|---|---|
| Cu | Cu | −1.8% | +3.4% | −0.8% | −11.3% | +14.0% | +0.4% |
| W | W | +4.2% | +8.5% | +19.4% | −2.2% | +22.2% | +8.6% |
| Si | Si | +13.6% | +9.9% | −5.9% | −3.4% | +27.1% | +26.2% |
| NaCl | Na | −11.5% | +0.7% | +7.1% | −21.5% | +13.4% | −9.2% |
| NaCl | Cl | −13.3% | −6.7% | +3.9% | −26.1% | +13.7% | −8.5% |
| MgO | Mg | +3.4% | +10.8% | +96.5% | +4.9% | +78.1% | +22.5% |
| MgO | O | −8.2% | −1.9% | +72.8% | −5.0% | +63.1% | +12.5% |
| ZnO | Zn | +5.8% | +13.5% | +26.1% | +16.5% | +35.1% | +23.3% |
| ZnO | O | −11.4% | −5.3% | +8.0% | −1.1% | +11.8% | +4.5% |
| Mean absolute | | 8.1% | 6.7% | 26.7% | 10.2% | 30.9% | 12.9% |

Difference of $U_\mathrm{iso}$ at 300 K from experiment.

MACE-OMAT-0 is closest to experiment, with a mean absolute difference of 6.7%, and
pheasy follows with 8.1%. The two PBE potentials give $U$ 63% to 97% too large for
MgO.

The NAC changes the frequencies near Gamma. The next cell gives the largest
relative change of an eigenvalue at 300 K for each material with Born charges in
the database.

In [ ]:
nac_change = {}
for name, entry in mlip_materials.items():
    values = entry["eigenvalues"]
    if "pheasy, NAC" not in values:
        continue
    no_nac = np.array(values["pheasy"])[i300]
    nac = np.array(values["pheasy, NAC"])[i300]
    nac_change[name] = 100 * np.abs(nac / no_nac - 1).max()
pd.Series(nac_change, name="largest change with the NAC (%)").round(2)

| Material | Largest change with the NAC |
|---|---|
| Cu, W, Si | 0.00% |
| NaCl | 0.04% |
| MgO | 0.05% |
| ZnO | 6.17% |
| UO3 | 35.37% |
| KTe | 6.84% |
| CaYAlO4 | 20.74% |
| Ba3In2O5F2 | 105.08% |
| RbLaSiS4 | 66.66% |
| Ba3Sr3(SbN)2 | 1.46% |
| TlCu3S2 | 38.81% |
| CsHo2Cu3Se5 | 2656.88% |

The NAC changes $U$ by less than 0.1% for NaCl and MgO, and by 6% for ZnO. For the
materials with imaginary modes it changes $U$ by up to a factor of 27. Their $U$
comes mostly from the real modes just above 0.1 THz, and the NAC moves these
modes. $U$ from the real modes of an unstable material is therefore only a rough
estimate.

AgBr has no pheasy force constants. The next cell gives $U_\mathrm{iso}$ of Ag and
Br at 300 K for each potential, the mean of the three eigenvalues.

In [ ]:
agbr = mlip_materials["AgBr"]
agbr_species = np.array(agbr["species"])
pd.DataFrame(
    {
        potential: {
            element: np.array(agbr["eigenvalues"][potential])[i300][
                agbr_species == element
            ].mean()
            for element in dict.fromkeys(agbr["species"])
        }
        for potential in POTENTIALS
    }
).round(5)

At 300 K the five potentials give $U_\mathrm{iso}$ from 0.032 to 0.050 Å² for Ag and
from 0.027 to 0.076 Å² for Br. Without a DFT reference, the spread between the
potentials is the only check of these numbers.

### B8. Imaginary modes

The next cell plots the eigenvalues of the pheasy reference with "absolute" against
those from the real modes only, for all sites of the materials with imaginary
modes in the reference, at all temperatures, on log axes.

In [ ]:
ref, absolute, temp = [], [], []
for entry in mlip_materials.values():
    if entry["imaginary_modes"].get("pheasy") != "exclude":
        continue
    values = np.array(entry["eigenvalues"]["pheasy"])
    ref.append(values.ravel())
    absolute.append(np.array(entry["eigenvalues"]["pheasy, absolute"]).ravel())
    temp.append(np.repeat(temperatures, values[0].size))
ref, absolute, temp = map(np.concatenate, (ref, absolute, temp))
_, ax = plt.subplots(figsize=(5, 4.5))
points = ax.scatter(ref, absolute, c=temp, cmap=t_cmap, vmin=0, vmax=1000, s=8)
# log axes, since a few sites are far larger than the rest
low, top = 0.8 * min(ref.min(), absolute.min()), 1.25 * max(ref.max(), absolute.max())
ax.plot([low, top], [low, top], color="0.5", lw=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(low, top)
ax.set_ylim(low, top)
ax.set_aspect("equal")
ax.set_xlabel("pheasy, real modes (Å$^2$)")
ax.set_ylabel('pheasy, "absolute" (Å$^2$)')
plt.colorbar(points, ax=ax, label="Temperature (K)")
plt.tight_layout()
plt.show()
{
    "MAE (A^2)": round(float(np.abs(absolute - ref).mean()), 5),
    "median relative difference (%)": round(
        float(100 * np.median(absolute / ref - 1)), 1
    ),
    "largest ratio": round(float((absolute / ref).max()), 1),
}

With "absolute", the eigenvalues are 4.6% larger than from the real modes only in
the median, and the MAE is 0.020 Å². For single sites the difference is much
larger. At 300 K, one site of CsHo2Cu3Se5 has an eigenvalue 67 times larger, of
1.36 Å². The imaginary modes have no restoring force, so these values depend on
how unstable the modes are, not on the vibrations of the crystal. All eigenvalues
are positive at 0 K.

KTe has imaginary modes in the pheasy reference. The next cell plots
$\log_{10}(I_T/I_0)$ of its XRD peaks at each temperature, with $U$ of the pheasy
reference from the real modes and with "absolute", and with $U$ of MACE-OMAT-0.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True, layout="constrained")
for ax, source in zip(axes, ("pheasy", "pheasy, absolute", "MACE-OMAT-0"), strict=True):
    pattern = mlip_materials["KTe"]["xrd"][source]
    two_theta = np.array(pattern["two_theta"])
    static = np.array(pattern["intensity_static"])
    keep = static > 1e-3 * static.max()
    for t, intensity in zip(temperatures, pattern["intensity"], strict=True):
        ratio = np.log10(np.array(intensity)[keep] / static[keep])
        ax.plot(two_theta[keep], ratio, "o", ms=2.5, color=t_cmap(t / 1000))
    ax.set_title(f"KTe, {source}")
    ax.set_xlabel("2$\\theta$ (°)")
axes[0].set_ylabel("$\\log_{10}(I_T/I_0)$")
fig.colorbar(
    plt.cm.ScalarMappable(cmap=t_cmap, norm=plt.Normalize(0, 1000)),
    ax=axes,
    label="Temperature (K)",
)
plt.show()

The median of $\log_{10}(I_T/I_0)$ at 1000 K is −1.59 with the real modes of
pheasy, −1.68 with "absolute" and −1.57 with MACE-OMAT-0. The pheasy values
scatter more. The real modes next to the imaginary ones make $U$ of some sites
large, up to 0.053 Å² at 300 K against 0.037 Å² with MACE-OMAT-0. With
"absolute" the largest $U$ of KTe is 0.085 Å².

## Known limitations

- $U$ is harmonic. The frequencies are those of the structure relaxed at 0 K, and
  are not renormalized with temperature. Thermal expansion and anharmonic effects
  are left out, and these grow with temperature.
- On a finite q-point mesh, $U$ is too small by an error that falls as $1/N$
  (B6).
- If the q-point mesh has imaginary modes, the phonon workflow stores no $U$, and
  the Debye-Waller workflow fails. With
  `imaginary_modes_thermal_displacements="exclude"`, $U$ comes from the real modes
  above `freq_min_thermal_displacements`. With "absolute", each imaginary mode
  counts as a real mode of the same absolute frequency. Both are only estimates
  and depend strongly on the modes near zero frequency (B7, B8).
- The comparison of the potentials has only six stable materials.
- The Born charges and the dielectric tensor come from MACE-Field.
- The references for Cu and W are computed from measured phonon densities of
  states. The reference for NaCl comes from a structure refinement, not from a
  dedicated measurement of the displacements.